# SmartCampus AI — évaluation des LLM sur GPU Kaggle / Colab (SA-31)

Même protocole que `llm-eval/evaluate.py` en local, mais sur un GPU T4 (16 Go) pour tester des modèles 7–9B.

1. **Kaggle** : *Settings → Accelerator → GPU T4*, *Internet → On*. **Colab** : *Exécution → Modifier le type d'exécution → GPU T4*.
2. Préparer sur votre PC une archive `smartcampus-eval.zip` contenant `llm-eval/` et `backend/app/` (commande en bas de ce notebook), puis l'importer (Kaggle : *Add data → Upload* ; Colab : panneau Fichiers).
3. Exécuter les cellules dans l'ordre, puis télécharger `results.zip`.

Aucune donnée personnelle n'est envoyée : seulement le jeu de questions et le prompt système.

In [ ]:
# 1) Installer Ollama et les dépendances Python du banc
!curl -fsSL https://ollama.com/install.sh | sh
!pip -q install httpx pydantic-settings

In [ ]:
# 2) Démarrer le serveur Ollama en arrière-plan
import subprocess, time, httpx
server = subprocess.Popen(["ollama", "serve"], stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(30):
    try:
        httpx.get("http://localhost:11434/api/tags", timeout=2); print("Ollama prêt"); break
    except Exception:
        time.sleep(1)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 3) Décompresser le code (adapter le chemin : Kaggle = /kaggle/input/<dataset>/smartcampus-eval.zip)
import glob, os
zip_path = (glob.glob("/kaggle/input/*/smartcampus-eval.zip") + glob.glob("/content/smartcampus-eval.zip") + ["smartcampus-eval.zip"])[0]
!unzip -oq "$zip_path" -d /tmp/smartcampus
os.chdir("/tmp/smartcampus/llm-eval")
!wc -l dataset.jsonl

In [ ]:
# 4) Modèles à comparer (open-source, adaptables en LoRA). Sur T4 16 Go, les 7–9B en Q4 passent.
MODELS = ["qwen2.5:7b-instruct", "llama3.1:8b", "gemma2:9b", "aya-expanse:8b"]
for m in MODELS:
    !ollama pull {m}
!ollama list

In [ ]:
# 5) Lancer le banc (≈ 66 questions × 2 appels × 4 modèles)
models = " ".join(MODELS)
!LLM_PROVIDER=openai python evaluate.py --models {models} --base-url http://localhost:11434/v1

In [ ]:
# 6) Résultats : afficher le tableau et préparer le téléchargement
print(open("results/summary.md", encoding="utf-8").read())
!nvidia-smi --query-gpu=name --format=csv,noheader > results/gpu.txt
!zip -qr /tmp/results.zip results && cp /tmp/results.zip /kaggle/working/ 2>/dev/null || cp /tmp/results.zip /content/ 2>/dev/null; ls -la /tmp/results.zip

## Préparer l'archive sur votre PC

Depuis la racine du dépôt (Git Bash) :

```bash
zip -r smartcampus-eval.zip llm-eval backend/app -x '*/__pycache__/*' 'llm-eval/results/*'
```

PowerShell : `Compress-Archive -Path llm-eval, backend\app -DestinationPath smartcampus-eval.zip`

Ensuite, copier `results/` dans `llm-eval/results/` sur le PC et remplir `docs/choix-llm.md` **uniquement avec les chiffres mesurés** (préciser le GPU : les latences dépendent de la machine).